# Netflix Prize — Personalized Content Discovery (Walkthrough)

End-to-end narrative: **EDA → models → evaluation (RMSE & MAP@10) → recommendations**.

Run the pipeline scripts first (`01`→`05`) so the cached artifacts exist, then run this notebook top to bottom.
All models are implemented from scratch in NumPy (see `src/`).


## 0. Setup


In [ ]:
import os, sys, json
ROOT = os.path.abspath('..')                      # repo root (run from notebooks/)
sys.path.insert(0, ROOT); sys.path.insert(0, os.path.join(ROOT, 'src'))
os.environ.setdefault('NETFLIX_DATA_DIR', os.path.dirname(ROOT))
import numpy as np, pandas as pd
import config, data_processing as dp, evaluation as ev, recommend as rc
from models import BaselineModel, SVDModel, ItemCFModel, PopularityModel, PopBlendModel

ds = dp.Dataset.load(os.path.join(config.PROCESSED_DIR, 'dataset.npz'))
titles = dp.load_movie_titles()
print(f'{ds.n_users:,} users x {ds.n_items:,} movies | train {len(ds.train_r):,} | test {len(ds.test_r):,}')


## 1. Exploratory Data Analysis

Computed on `combined_data_1.txt` (24.1M ratings) to reflect the *unfiltered* catalogue.

- **470,758 users · 4,499 movies · 98.86% sparse**
- **Mean rating ≈ 3.6**, strong positivity bias (66% of ratings are 4–5★)
- **Heavy-tailed user activity**: median 24 ratings/user, max 4,467
- **Extreme content long tail**: top 20% of movies get **89%** of all ratings


In [ ]:
print(json.dumps(json.load(open(os.path.join(config.RESULTS_DIR, 'eda_stats.json'))), indent=2))


![rating distribution](../outputs/figures/rating_distribution.png)
![user activity](../outputs/figures/user_activity.png)
![movie popularity](../outputs/figures/movie_popularity.png)
![ratings over time](../outputs/figures/ratings_over_time.png)


## 2. Models

| Model | Idea |
|---|---|
| Baseline | `mu + b_u + b_i` regularized biases |
| SVD | biased matrix factorization (Funk-SVD, SGD) |
| Item-CF | centered-cosine item-item neighborhood |
| Popularity | most-rated movies (reference) |
| Hybrid | SVD + popularity blend |


## 3. Evaluation — RMSE & MAP@10

Per-user 20% hold-out. Relevance = actual rating ≥ 3.5. MAP@10 over all unseen items.


In [ ]:
df = pd.read_csv(os.path.join(config.RESULTS_DIR, 'metrics.csv'), index_col=0)
df.round(4)


**Reading the table**

1. SVD & Item-CF win **RMSE** (~0.89 vs 0.92 baseline).
2. **Rating accuracy ≠ ranking**: SVD's MAP@10 is among the worst — ranking by
   predicted rating surfaces niche high-rated titles.
3. **Item-CF is the best all-rounder** (top RMSE *and* MAP@10 *and* coverage 0.81).
4. The **hybrid lifts SVD's MAP@10 ~20×** (0.005→0.105) at identical RMSE.


## 4. Recommendations

Top-10 = score all movies, drop those already rated in train, take the top 10.
A "HIT" is a recommended movie that is in the user's held-out set with rating ≥ 3.5.


In [ ]:
idx = rc.build_user_index(ds)
icf = ItemCFModel().load(os.path.join(config.MODEL_DIR, 'itemcf.npz'))
icf.attach_profiles(ds.train_u, ds.train_i, ds.train_r, ds.n_users)
svd = SVDModel().load(os.path.join(config.MODEL_DIR, 'svd.npz'))

u = 37843
print('LIKES:')
for t, r in rc.liked_history(ds, u, titles, idx, 6): print(f'  {r:.0f}*  {t}')
print('\nITEM-CF Top-10:')
for n, rec in enumerate(rc.recommend(icf, u, ds, titles, idx, 10), 1):
    print(f"  {n:2d}. {rec['title']}  ({rec['score']})" + ('  <-- HIT' if rec['hit'] else ''))


### 'More like this' — item-item similarity (Item-CF)


In [ ]:
def find_idx(sub):
    for i, raw in enumerate(ds.raw_movie_ids):
        if sub.lower() in titles.get(int(raw), '').lower(): return i
    return None

for name in ['American Beauty', 'Sixth Sense', 'Lord of the Rings: The Fellowship']:
    i = find_idx(name)
    if i is not None:
        print(titles[int(ds.raw_movie_ids[i])], '->')
        for t, s in rc.similar_movies(icf, i, ds, titles, 6): print(f'    {s:.3f}  {t}')
        print()


## 5. Conclusions

- Collaborative methods substantially beat the bias baseline on **rating prediction**.
- **Item-based CF** gives the best overall recommendations: strong RMSE, the best
  MAP@10, the most diverse coverage, and coherent, explainable 'more like this' lists.
- Offline Top-N is **popularity-dominated**; a popularity blend is a cheap, strong
  ranking booster (the SVD→Hybrid jump), illustrating the accuracy↔ranking trade-off.
- **Future work**: full-data training, learning-to-rank (BPR/WARP), metadata-aware
  hybrids for cold-start, and neural CF.
